In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredatabricks_catalog.bronze

# Data Reading

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### Customers Data

In [0]:
df_customers = spark.read.format("csv")\
                .option("header", True)\
                .option("inferSchema", True)\
                .load("abfss://raw@azuredatabricksansh.dfs.core.windows.net/staging/customers")

display(df_customers)

In [0]:
df_customers.schema

In [0]:
## StrcutType Schema
my_struct_schema = StructType([StructField('customer_id', IntegerType(), True), StructField('customer_name', StringType(), True), StructField('email', StringType(), True), StructField('country', StringType(), True), StructField('signup_date', DateType(), True), StructField('customer_type', StringType(), True), StructField('age', DoubleType(), True), StructField('updated_at', TimestampType(), True)])


## DDL Schema
my_ddl_schema = """
cutomer_id STRING,
customer_name STRING,
email STRING,
country STRING,
signup_date DATE,
customer_type STRING,
age DOUBLE,
updated_at TIMESTAMP
"""


In [0]:
df_customers = spark.read.format("csv")\
                .option("header", True)\
                .schema(my_struct_schema)\
                .load("abfss://raw@azuredatabricksansh.dfs.core.windows.net/staging/customers")

display(df_customers)

In [0]:
df_customers.write.format("delta")\
            .mode("append")\
            .saveAsTable("azuredatabricks_catalog.bronze.customers")

### Products

In [0]:
%sql
COPY INTO azuredatabricks_catalog.bronze.products
FROM 'abfss://raw@azuredatabricksansh.dfs.core.windows.net/staging/products'
FILEFORMAT = PARQUET
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')


### Orders Table

In [0]:
df_orders = spark.read.format("json")\
                .option("multiLine",False)\
                .option("inferSchema",True)\
                .load("abfss://raw@azuredatabricksansh.dfs.core.windows.net/staging/orders")

df_orders.write.format("delta")\
            .mode("append")\
            .saveAsTable("azuredatabricks_catalog.bronze.orders")